In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%pip install -q optuna scikit-learn pandas pyarrow mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
import sys
sys.path.append('/content/drive/MyDrive/repositories/aq_edge')

In [7]:
%cd /content/drive/MyDrive/repositories/aq_edge/colab
%run mlflow_colab.py

/content/drive/MyDrive/repositories/aq_edge/colab
MLflow server ready: http://127.0.0.1:5000


In [23]:
%cd /content/drive/MyDrive/repositories/aq_edge

import os
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"

!python hyperparameter_optuna.py --optuna.n_trials 2 --optuna.include_target_history false

/content/drive/MyDrive/repositories/aq_edge/colab
[2026-10-21 07:08:35] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/preprocessing_20261021_070835.log
[2026-10-21 07:08:35] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/metrics_20261021_070835.log
[2026-10-21 07:08:38] INFO: Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261021_070838.log
INFO:__main__:Logging to file: /content/drive/MyDrive/repositories/aq_edge/artifacts/main_20261021_070838.log
[2026-10-21 07:08:38] INFO: Starting CO2 forecasting hyperparameter optimization
INFO:__main__:Starting CO2 forecasting hyperparameter optimization
[2026-10-21 07:08:38] INFO: Device: cpu
INFO:__main__:Device: cpu
[2026-10-21 07:08:38] INFO: Station: UTEC
INFO:__main__:Station: UTEC
[2026-10-21 07:08:38] INFO: Features: ['CO2', 'TEM', 'HUM', 'VOC']
INFO:__main__:Features: ['CO2', 'TEM', 'HUM', 'VOC']
[2026-10-21 07:08:38] INFO: Target: CO2
INFO:__main_

In [ ]:
import os
import shutil
import sqlite3

os.makedirs(BACKUP, exist_ok=True)

# Consistent SQLite backup
src = sqlite3.connect(DB)
dst = sqlite3.connect("/content/mlflow_snapshot.db")
with dst:
    src.backup(dst)
src.close()
dst.close()
shutil.copy2("/content/mlflow_snapshot.db", f"{BACKUP}/mlflow.db")

# Preserve experiment artifacts
if os.path.isdir(ARTIFACTS):
    shutil.copytree(
        ARTIFACTS, f"{BACKUP}/mlartifacts", dirs_exist_ok=True
    )

print("MLflow database and artifacts backed up to Drive.")

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone

import optuna

output_dir = Path("optuna_results")
output_dir.mkdir(parents=True, exist_ok=True)

# Save all trials
study.trials_dataframe().to_csv(
    output_dir / "trials.csv",
    index=False,
)

# Save the best trial
if study.best_trial is not None:
    result = {
        "study_name": study.study_name,
        "best_trial": study.best_trial.number,
        "objective": "validation_r2",
        "best_value": study.best_value,
        "params": study.best_trial.params,
        "saved_at": datetime.now(timezone.utc).isoformat(),
    }

    with (output_dir / "best_params.json").open("w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Results saved to {output_dir.resolve()}")